In [1]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from numpy import random
import cv2
from numpy import identity

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
print(os.listdir('/content/drive/MyDrive'))

['General presentation.gslides', 'SLIDE PRESENTATION:.gdoc', 'Photos', 'Em2.1bexbct.gdoc', 'amankhanal.mp4', 'filter3.1bei.gdoc', 'landslide4sense.zip', 'best_unet_model.pth']


In [5]:
!unzip -q "/content/drive/MyDrive/landslide4sense.zip" -d /content/

In [6]:
# =============================================================================
# HELPER FUNCTION: Compute Topographic Features from DEM
# =============================================================================
# This is a plain Python function — no PyTorch needed here.
# It takes numpy arrays as input and returns numpy arrays as output.


def compute_topographical_features(dem, slope_deg, res=10.0, azimuth=315, angle_altitude=45):
    """ Compute northness, eastness, profile curvature"""

    dem_padded = np.pad(dem, pad_width=1, mode="edge")

    dy, dx = np.gradient(dem_padded, res)

    d2y, _ = np.gradient(dy, res)
    _, d2x = np.gradient(dx, res)

    # removing the padding added in the beginning
    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)
    eastness = np.sin(aspect)

    curvature = d2x + d2y

    return northness, eastness, curvature


def compute_normalization(img_dir, file_ids): 
    """ Compute the mean and the standard deviation from the training set only. Protected against NaN vlaues. """
    N_CHANNELS = 18
    channel_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    pixel_count = 0
    eps = 1e-6
    
    for file_id in file_ids:
        img_path = os.path.join(img_dir, f"image_{file_id}.h5")
        if not os.path.exists(img_path):
            continue  # Skip if the file does not exist

        with h5py.File(img_path, "r") as f:
            raw_image = f["img"][:]
            
        blue  = raw_image[:, :, 1].astype(np.float32)
        green = raw_image[:, :, 2].astype(np.float32)
        red   = raw_image[:, :, 3].astype(np.float32)
        b5    = raw_image[:, :, 4].astype(np.float32)
        b6    = raw_image[:, :, 5].astype(np.float32)
        b7    = raw_image[:, :, 6].astype(np.float32)
        nir   = raw_image[:, :, 7].astype(np.float32)
        b8a   = raw_image[:, :, 8].astype(np.float32)
        swir1 = raw_image[:, :, 10].astype(np.float32)
        swir2 = raw_image[:, :, 11].astype(np.float32)
        slope = raw_image[:, :, 12].astype(np.float32)
        dem   = raw_image[:, :, 13].astype(np.float32)
        
        northness, eastness, curvature = compute_topographical_features(dem, slope)
        
        ndvi = (nir - red) / (nir + red + eps)
        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
        ndwi = (green - nir) / (green + nir + eps)
        
        image_18ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, b8a, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        )  # final 18 channel raster
        
        image_18ch = np.nan_to_num(image_18ch, nan=0.0)  # Replace NaN values with 0.0
        
        h, w, _ = image_18ch.shape
        
        channel_sum += np.sum(image_18ch, axis=(0, 1))
        channel_squared_sum += np.sum(image_18ch ** 2, axis=(0, 1))
        pixel_count += h * w

    means = channel_sum / pixel_count
    stds = np.sqrt((channel_squared_sum / pixel_count) - (means ** 2))
    
    # At the end, it spits out two lists. Each list has exactly 18numbers in it.

    # self.means = [mean_ch1, mean_ch2, mean_ch3, ..., mean_ch18]

    # self.stds = [std_ch1, std_ch2, std_ch3, ..., std_ch18]
        

    return means.astype(np.float32), stds.astype(np.float32)

def train_transform(means, stds):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=(-0.1, 0.1),
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5
        ),
        # Applying normalization here
        # it does img = (img - mean * max_pixel_value) / (std * max_pixel_value) inder the hood
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),

        ToTensorV2()
    ])


def val_transform(means, stds):
    return A.Compose([
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),
        ToTensorV2()
    ])


# =============================================================================
# PYTORCH DATASET CLASS: LandslideDataset
# =============================================================================
# In TensorFlow you often loaded data with tf.data.Dataset or keras utilities.
# In PyTorch, the standard way is to define a Dataset class with these 3 methods:
#
#   __init__  → runs once when you create the dataset object (setup/configuration)
#   __len__   → tells PyTorch how many samples exist (used by DataLoader)
#   __getitem__ → loads and returns ONE sample by index (called repeatedly during training)
#
# PyTorch's DataLoader will call __getitem__ automatically in batches.


class LandslideDataset(Dataset):

    def __init__(self, img_dir, mask_dir=None, transform=None, file_ids = None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        if file_ids is not None:
            self.file_ids = file_ids
        else:
            self.file_ids = sorted(
                [
                    int(f.split("_")[1].split(".")[0])
                    for f in os.listdir(img_dir)
                    if f.endswith(".h5")
                ]
            )

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        img_name = f"image_{file_id}.h5"
        mask_name = f"mask_{file_id}.h5"

        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)

        eps = 1e-6

        blue = raw_image[:, :, 1]
        green = raw_image[:, :, 2]
        red = raw_image[:, :, 3]
        b5 = raw_image[:, :, 4]
        b6 = raw_image[:, :, 5]
        b7 = raw_image[:, :, 6]
        nir = raw_image[:, :, 7]
        b8a = raw_image[:, :, 8]
        swir1 = raw_image[:, :, 10]
        swir2 = raw_image[:, :, 11]
        
        # Terrain features
        slope = raw_image[:, :, 12]
        dem = raw_image[:, :, 13]

        northness, eastness, curvature = compute_topographical_features(dem, slope)

        ndvi = (nir - red) / (nir + red + eps)

        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)

        ndwi = (green - nir) / (green + nir + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 18)
        image_18ch = np.stack(
            [dem, slope, northness, eastness, curvature, blue, green, red, nir, b5, b6, b7, b8a, swir1, swir2, ndvi, bsi, ndwi], axis=-1
        ).astype(np.float32)  # Final shape: (128, 128, 18)

        image_18ch = np.nan_to_num(image_18ch, nan=0.0, posinf=0.0, neginf=0.0)

    # this will do the normalization, rotation and convert to the tensors
        if self.transform:

            augmented = self.transform(image=image_18ch, mask=mask)
            image = augmented['image'].float()
            mask = augmented['mask'].long()
        else:
            image_18ch = image_18ch.transpose((2, 0, 1))  # (C, H, W)

            image = torch.from_numpy(image_18ch).float()
            mask = torch.from_numpy(mask).long()
            

        return image, mask


In [7]:
# # ----- Part-1: U-Net Architecture -----


# class ConvBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.block = nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),  ## out_channels means filters in keras, and keras figures out the in_channel itself
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#         )

#     def forward(self, x):
#         return self.block(x)


# # Encoder Block: ConvBlock + MaxPool


# class EncoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv = ConvBlock(in_channels, out_channels)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

#     def forward(self, x):
#         features = self.conv(x)
#         pooled = self.pool(features)
#         return features, pooled
#         # return both:
#         # features -> will be passed accross via skip connection to decoder
#         # pooled -> goes down to the next encoder block


# # Decoder Block: Upsampe + Concatenate skip + convBlock
# class DecoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.upsample = nn.ConvTranspose2d(
#             in_channels, out_channels, kernel_size=2, stride=2
#         )
#         # convTransposed2d doubles the spatial size: (64, 64) → (128, 128)

#         self.conv = ConvBlock(out_channels * 2, out_channels)

#     def forward(self, x, skip):
#         x = self.upsample(x)
#         x = torch.cat([x, skip], dim=1)  # concatenate along channel dimension
#         x = self.conv(x)
#         return x


# # --- Full U-Net Model ---
# class UNet(nn.Module):
#     def __init__(self, in_channels=8, num_classes=2):
#         """
#         in_chennels : number of input channels - 8 for our dataset
#         num_classes : 2 for binary segmentation (landslide / no-landslide)
#         """

#         super().__init__()

#         # Encoder ( Contracting Path)

#         self.enc1 = EncoderBlock(in_channels, 64)  # 8 → 64
#         self.enc2 = EncoderBlock(64, 128)  # 64 → 128
#         self.enc3 = EncoderBlock(128, 256)  # 128 → 256
#         self.enc4 = EncoderBlock(256, 512)  # 256 → 512

#         # Bottleneck (deepest point - no pooling)

#         self.bottleneck = ConvBlock(512, 1024)  # 512 → 1024

#         # Decoder (Expanding Path)

#         self.dec4 = DecoderBlock(1024, 512)  # 1024 → 512
#         self.dec3 = DecoderBlock(512, 256)  # 512 → 256
#         self.dec2 = DecoderBlock(256, 128)  # 256 → 128
#         self.dec1 = DecoderBlock(128, 64)  # 128 → 64

#         # --- Final Output Layer ---
#         self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)
#         # kernel size=1 -> 1x1 convolution, just maps 64 channels -> num_classes

#     def forward(self, x):
#         # ---Encoder ---
#         skip1, x = self.enc1(x)  # skip for dec1, x: goes to enc2
#         skip2, x = self.enc2(x)
#         skip3, x = self.enc3(x)
#         skip4, x = self.enc4(x)  # skip for dec4, x: goes to bottleneck

#         # --- Bottleneck ---
#         x = self.bottleneck(x)

#         # --- Decoder ---
#         x = self.dec4(x, skip4)  # input from bottleneck, skip from enc4
#         x = self.dec3(x, skip3)
#         x = self.dec2(x, skip2)
#         x = self.dec1(x, skip1)  # input from dec2, skip from enc1

#         # --- Final Output ---
#         return self.output_conv(x)  # shape: (Batch, 2, 128, 128)

In [8]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection
        # if the input and the output dont match, we need a 1x1 convolution,
        # to project the correct input before adding.
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity
        out = self.relu(out)
        
        return out


# ---- Encoder Block ---- #

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled

# ---- Decoder Block ---- #

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )

        self.conv = ResidualBlock(out_channels * 2, out_channels)

    def forward(self, x , skip):
        upsampled = self.upsample(x)
        cat = torch.cat([upsampled, skip], dim=1)
        x = self.conv(cat)
        return x
    
class ResUNet(nn.Module):
    def __init__(self, in_channels=18, num_classes=2):
        super().__init__()

        # --Encoding Phase -- 

        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # --Bottleneck (deepest point - no pooling here)

        self.bottleneck = ResidualBlock(512, 1024)

        # --Decoding Phase --

        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        # -- Final Outout Layer -- 
        self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
            # ---Encoder---
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # ---Bottleneck---
        x = self.bottleneck(x)

        #---Decoder---
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final output

        return self.output_conv(x)

In [9]:
import torch
import torch.nn as nn

# =============================================================================
# PART 2: LOSS FUNCTION
# =============================================================================
# BACKGROUND CONTEXT:
# In landslide segmentation, we face extreme "Class Imbalance". A satellite patch 
# might have 16,000 background pixels and only 200 landslide pixels. 
# Standard CrossEntropy treats every pixel equally, which tempts the model to 
# lazily guess "No Landslide" everywhere to achieve 99% accuracy. 
# Dice Loss fixes this by ignoring background pixels and forcing the model to 
# focus entirely on maximizing the regional overlap (Intersection) with actual landslides.
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        # self.smooth prevents a catastrophic "Division by Zero" error.
        # If an image has 0 landslide pixels and the model correctly predicts 0,
        # the formula becomes 0/0 (which crashes). Adding 1.0 to top and bottom stabilizes it.
        self.smooth = smooth
 
    def forward(self, predictions, targets):
        # ---------------------------------------------------------------------
        # INPUT SHAPES:
        # predictions: (Batch, 2, H, W) -> Raw network outputs (Logits like [-2.4, 3.1])
        # targets:     (Batch, H, W)    -> Ground truth mask containing integers (0 or 1)
        # ---------------------------------------------------------------------
 
        # STEP 1: CONVERT LOGITS TO PROBABILITIES AND DROP THE BACKGROUND CHANNEL
        # 1. torch.softmax(..., dim=1) takes the raw channel values for every pixel 
        #    and crushes them into percentages (0.0 to 1.0) that sum to exactly 1.0 (100%).
        # 2. [:, 1, :, :] uses standard Python tuple indexing. It says:
        #    - First ':'  -> Keep all images in the batch.
        #    - '1'        -> Grab only Channel index 1 (the Landslide channel).
        #    - Last ':,:' -> Keep all Height rows and Width columns.
        # 3. DIMENSION DROPPING: Because we indexed with a single integer ('1') instead
        #    of a range ('1:2'), PyTorch drops the channel dimension entirely.
        #
        # FINAL SHAPE: (Batch, H, W) 
        # WHAT IS INSIDE: The decimal number stored inside each cell *is* the probability (0.0-1.0).
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :] 
        
        # Convert integer targets (0 and 1) to floats so we can do decimal multiplication with probs
        targets_f = targets.float()
 
        # STEP 2: CALCULATE THE INTERSECTION (OVERLAP)
        # Multiplying 'probs' (0.0 to 1.0) by 'targets_f' (0 or 1) acts as a mask.
        # If the target pixel is 0 (Background), the product becomes 0.
        # If the target pixel is 1 (Landslide), it preserves the model's probability score.
        # .sum(dim=(1, 2)) flattens and sums all pixel values across Height and Width.
        # OUTPUT SHAPE: (Batch,) -> One total intersection score per image in the batch.
        intersection = (probs * targets_f).sum(dim=(1, 2))
 
        # STEP 3: THE DICE COEFFICIENT FORMULA
        # Dice = (2 * |Intersection|) / (|Predictions| + |Targets|)
        # This measures the percentage of overlap between the predicted map and real map.
        # A perfect match yields a Dice Score of 1.0. Complete failure yields 0.0.
        # OUTPUT SHAPE: (Batch,)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        
        # STEP 4: RETURN THE LOSS
        # Deep learning optimizers are built to MINIMIZE a loss value toward 0.
        # Since we want to MAXIMIZE our Dice Score toward 1.0, we calculate Loss = 1 - Dice.
        # .mean() averages this loss score across all images in the batch.
        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    """Focal Loss for binary segmentation (landslide vs background).
       Down-weights easy background pixels and focuses on hard, rare landslide pixels."""
    def __init__(self, alpha=0.75, gamma=2.0, smooth=1e-6):
        # alpha = weight for positive class (landslide). Larger alpha = more focus on landslides.
        # gamma = focusing parameter; typical range [0.5, 5.0]. Higher gamma = more aggressive down-weighting of easy examples.
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth
        
    def forward(self, logits, targets):
        # logits: (B,2,H,W) raw outputs; targets: (B,H,W) with 0/1
        probs = torch.softmax(logits, dim=1)[:, 1]      # probability of landslide
        targets_f = targets.float()
        
        # p_t = probability of the true class (landslide if target==1, else 1-prob)
        pt = torch.where(targets_f == 1, probs, 1 - probs)
        
        # alpha_t = class‑dependent balancing factor: alpha for landslide, 1-alpha for background
        alpha_t = torch.where(targets_f == 1, self.alpha, 1 - self.alpha)
        
        # Focal loss per pixel: -α_t * (1-p_t)^γ * log(p_t)
        focal = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + self.smooth)
        
        return focal.mean()


class CombinedFocalDiceLoss(nn.Module):
    """Blend of Focal Loss (handles class imbalance) and Dice Loss (maximises region overlap).
       Often outperforms CE+Dice for extremely imbalanced tasks like landslide detection."""
    def __init__(self, focal_weight=0.5, dice_weight=0.5, alpha=0.75, gamma=2.0):
        super().__init__()
        self.focal = BinaryFocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        return (self.focal_weight * self.focal(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


class CombinedLoss(nn.Module):
    """Original combined loss: CrossEntropy + Dice.
       CE provides stable gradients early; Dice focuses on landslide overlap.
       This is a solid baseline before switching to Focal+Dice."""
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.dice = DiceLoss()
        
        # PyTorch's CrossEntropyLoss automatically accepts raw Logits (Batch, 2, H, W)
        # and integer Targets (Batch, H, W). It applies Softmax internally under the hood!
        self.ce   = nn.CrossEntropyLoss()
 
    def forward(self, predictions, targets):
        # WHY WE MIX THEM (50/50 Blend):
        # 1. CrossEntropy (CE) provides clean, highly stable mathematical gradients early 
        #    in training when the model is randomly guessing, but it suffers from class imbalance.
        # 2. Dice Loss ignores background pixels and focuses perfectly on landslide boundaries,
        #    but its gradients can be erratic and unstable when the model is completely untrained.
        # Combining them balances pixel-level confidence (CE) with global edge overlap (Dice).
        return (self.ce_weight   * self.ce(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))

In [10]:
# =============================================================================
# PART 3: METRICS
# =============================================================================
# What we track per epoch:
#   Loss    → how wrong the model is (lower is better)
#   IoU     → Intersection over Union — standard segmentation metric (higher is better)
#   F1      → same as Dice score — balances precision and recall (higher is better)
#   Precision → of all pixels predicted as landslide, how many actually are?
#   Recall    → of all actual landslide pixels, how many did we find?
 

def compute_metrics(predictions, targets, threshold=0.5):
    """
    predictions : (Batch, 2, H, W) raw logits from the model
    targets     : (Batch, H, W)    ground truth integer labels
    threshold   : float probability threshold for converting landslide probability to binary
    """
    # Convert logits to landslide probability (channel 1) and apply threshold
    probs = torch.softmax(predictions, dim=1)[:, 1]   # shape: (Batch, H, W)
    pred_bin = probs > threshold


    # True Positives, False Positives, False Negatives (with thresholded preds)
    tp = ((targets == 1) & (pred_bin == 1)).sum().float() 
    fp = ((targets == 0) & (pred_bin == 1)).sum().float()
    fn = ((targets == 1) & (pred_bin == 0)).sum().float()

    return tp, fp, fn

In [11]:

def train(img_dir, mask_dir, num_epochs, batch_size, lr, save_path):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # extracting and shuffling the file IDs
    all_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".h5")])
    all_ids = [int(f.split("_")[1].split(".")[0]) for f in all_files]
    
    # Shuffle with a fixed seed so train and val stays consistent
    random.seed(42)
    random.shuffle(all_ids)
    
    train_size = int(0.85 * len(all_ids))
    train_ids = all_ids[:train_size]
    val_ids = all_ids[train_size:]
    
    print(f"Total Samples: {len(all_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}")
    
    print("\n--- Computing Normalization Statistics From Training Split ---")
    MEANS, STDS = compute_normalization(img_dir, train_ids)
    
    train_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=train_transform(MEANS, STDS), file_ids = train_ids)
    
    val_dataset = LandslideDataset(img_dir=img_dir, mask_dir=mask_dir, transform=val_transform(MEANS, STDS), file_ids = val_ids)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size,  shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # after creating dataset and loader
    images, masks = next(iter(train_loader))
    print("Per-channel mean:", images.mean(dim=[0,2,3]))
    print("Per-channel std :", images.std(dim=[0,2,3]))
    
    
    # ----- Model, Loss and Optimizer ----- #
    model = ResUNet(in_channels=18, num_classes=2).to(device)
    # More aggressive — pushes the model harder toward finding landslides
    criterion = CombinedFocalDiceLoss(
        focal_weight=0.35,
        dice_weight=0.65,  
        alpha=0.80,       
        gamma=2.0
    )
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
    
    best_val_f1 = 0.0  # Track the best F1 score for saving the best model
    
    for epoch in range(1, num_epochs + 1):
        
        model.train()
        running_train_loss = 0.0
        
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()            # Clear old gradients
            predictions = model(images)      # Forward pass
            loss = criterion(predictions, targets) # Calculate loss
            loss.backward()                  # Backward pass (gradients)
            optimizer.step()                 # Update model weights
            
            running_train_loss += loss.item()
            
        train_loss = running_train_loss / len(train_loader)

        model.eval()
        running_val_loss = 0.0
        
        total_tp = 0
        total_fp = 0
        total_fn = 0
        
        with torch.no_grad(): # Disable gradient engine to save memory
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                
                predictions = model(images)
                loss = criterion(predictions, targets)
                running_val_loss += loss.item()
                
                # Calculate metrics for this batch using your dictionary utility
                batch_metrics = compute_metrics(predictions, targets)
                total_tp += batch_metrics[0]
                total_fp += batch_metrics[1]
                total_fn += batch_metrics[2]

        val_loss = running_val_loss / len(val_loader)
        
        # Step the learning rate scheduler based on validation loss
        scheduler.step(val_loss)
        
        # Average the metrics over all validation batches
        val_metrics = {
            "iou": total_tp / (total_tp + total_fp + total_fn + 1e-6),
            "f1": 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6),
            "precision": total_tp / (total_tp + total_fp + 1e-6),
            "recall": total_tp / (total_tp + total_fn + 1e-6)
        }

        # ---- Print Progress ----- #
        print(
            f"Epoch [{epoch:02d}/{num_epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f}  IoU: {val_metrics['iou']:.4f}  F1: {val_metrics['f1']:.4f}  "
            f"Precision: {val_metrics['precision']:.4f}  Recall: {val_metrics['recall']:.4f}"
            f"| LR: {optimizer.param_groups[0]['lr']:.6f}"
        )
        
    # Save the final trained model weights to your E: drive or local path
    current_val_f1 = val_metrics['f1']
    
    if current_val_f1 > best_val_f1:
        best_val_f1 = current_val_f1
    torch.save(model.state_dict(), save_path)
    
    
    print(f"\nTraining Complete! The best model achieved a Val F1 of {best_val_f1:.4f} and is saved at: {save_path}")
    
    return model


if __name__ == "__main__":
    # 1. Corrected Cloud Paths (Updated with the landslide4sense parent folder)
    IMG_DIR  = "/content/landslide4sense/TrainData/img"
    MASK_DIR = "/content/landslide4sense/TrainData/mask"
    
    # 2. Permanent Cloud Storage (Saves the model weights directly to your Google Drive)
    SAVE_PATH = "/content/drive/MyDrive/best_unet_model.pth"

    # 3. Hyperparameters
    BATCH_SIZE = 16  
    EPOCHS     = 65
    LEARNING_RATE = 1e-4

    print("--- Starting Cloud Landslide Mapping Pipeline ---")
    
    # 4. Kick off training
    trained_model = train(
        img_dir    = IMG_DIR,
        mask_dir   = MASK_DIR,
        num_epochs = EPOCHS,
        batch_size = BATCH_SIZE,
        lr         = LEARNING_RATE,
        save_path  = SAVE_PATH
    )

--- Starting Cloud Landslide Mapping Pipeline ---
Using device: cuda
Total Samples: 3799 | Train: 3229 | Val: 570

--- Computing Normalization Statistics From Training Split ---


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Per-channel mean: tensor([ 0.4528,  0.0579,  0.0603, -0.0018,  0.0044, -0.0756, -0.0499,  0.0023,
        -0.1224, -0.0199, -0.1081, -0.1325,  0.1009, -0.0317,  0.0323, -0.1220,
         0.0998,  0.0939])
Per-channel std : tensor([1.0697, 1.0093, 0.9987, 0.9827, 0.3079, 0.9300, 0.9193, 0.9188, 0.9774,
        0.9654, 0.9822, 0.9799, 1.0713, 1.0778, 1.1288, 1.0146, 1.0545, 1.0258])
Epoch [01/65] | Train Loss: 0.5230 | Val Loss: 0.4573  IoU: 0.4529  F1: 0.6234  Precision: 0.5013  Recall: 0.8244| LR: 0.000100
Epoch [02/65] | Train Loss: 0.4637 | Val Loss: 0.4201  IoU: 0.4096  F1: 0.5811  Precision: 0.4648  Recall: 0.7751| LR: 0.000100
Epoch [03/65] | Train Loss: 0.4105 | Val Loss: 0.3448  IoU: 0.5212  F1: 0.6852  Precision: 0.6212  Recall: 0.7639| LR: 0.000100
Epoch [04/65] | Train Loss: 0.3476 | Val Loss: 0.3190  IoU: 0.4415  F1: 0.6126  Precision: 0.5217  Recall: 0.7419| LR: 0.000100
Epoch [05/65] | Train Loss: 0.2690 | Val Loss: 0.2253  IoU: 0.5406  F1: 0.7018  Precision: 0.6718  Recal